# SoundStream Demo

https://github.com/serzai/neural-audio-codec

Author: Sergey Zaitsev

### 1. Environment setup

In [ ]:
!git clone https://github.com/serzai/neural-audio-codec.git
%cd neural-audio-codec

!pip install -r requirements.txt -q
# huggingface_hub here only for downloading checkpoint
!pip install huggingface_hub -q


### 2. Downloading checkpoint

In [ ]:
from huggingface_hub import hf_hub_download
import os

os.makedirs("saved", exist_ok=True)
CHECKPOINT_PATH = "saved/model_best.pth"

path = hf_hub_download(
    repo_id="serzai/neural-audio-codec",
    filename="checkpoint-epoch100.pth",
    local_dir="saved"
)

os.rename(path, CHECKPOINT_PATH)

### 3. Model initialization

In [ ]:
import torch
from src.model import SoundStreamModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = SoundStreamModel(
    in_channels=1,
    base_channels=32,
    latent_dim=128,
    strides=[2, 4, 5, 5],
    num_quantizers=8,
    codebook_size=1024
).to(device)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

### 4. Audio processing and comparison

In [ ]:
import torchaudio
import IPython.display as ipd
import matplotlib.pyplot as plt

# change url here
AUDIO_URL = "https://keithito.com/LJ-Speech-Dataset/LJ025-0076.wav"

def run_demo(url):
    # download
    !wget -q {url} -O input_audio.wav

    # load and preprocess
    waveform, sr = torchaudio.load("input_audio.wav")
    target_sr = 16000

    if sr != target_sr:
        waveform = torchaudio.transforms.Resample(sr, target_sr)(waveform)

    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    waveform = waveform - waveform.mean()
    if waveform.abs().max() > 0:
        waveform /= waveform.abs().max()

    # inference
    model.eval()
    with torch.no_grad():
        input_tensor = waveform.unsqueeze(0).to(device)
        output = model(input_tensor)
        reconstructed = output["fake_audio"].squeeze(0).cpu()

    # visualization
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    n_fft = 400
    hop_length = 160
    window = torch.hann_window(n_fft)

    # waveforms
    axes[0, 0].plot(waveform[0].numpy(), color="blue", alpha=0.7)
    axes[0, 0].set_title("Original waveform")
    axes[0, 1].plot(reconstructed[0].numpy(), color="green", alpha=0.7)
    axes[0, 1].set_title("Reconstructed waveform")

    # spectrograms
    spec_orig = torch.stft(waveform[0], n_fft=n_fft, hop_length=hop_length,
                          window=window, return_complex=True)
    spec_db_orig = 20 * torch.log10(torch.abs(spec_orig) + 1e-6)

    axes[1, 0].imshow(spec_db_orig.numpy(), aspect="auto", origin="lower",
                      extent=[0, waveform.shape[1]/target_sr, 0, target_sr/2],
                      cmap="viridis")
    axes[1, 0].set_title("Original spectrogram")
    axes[1, 0].set_ylabel("Frequency, Hz")

    spec_recon = torch.stft(reconstructed[0], n_fft=n_fft, hop_length=hop_length,
                           window=window, return_complex=True)
    spec_db_recon = 20 * torch.log10(torch.abs(spec_recon) + 1e-6)

    axes[1, 1].imshow(spec_db_recon.numpy(), aspect="auto", origin="lower",
                      extent=[0, reconstructed.shape[1]/target_sr, 0, target_sr/2],
                      cmap='viridis')
    axes[1, 1].set_title("Reconstructed spectrogram")
    axes[1, 1].set_ylabel("Frequency, Hz")

    plt.tight_layout()
    plt.show()

    # audio
    print("Original audio:")
    display(ipd.Audio(waveform[0].numpy(), rate=target_sr))
    print("Reconstructed audio:")
    display(ipd.Audio(reconstructed[0].numpy(), rate=target_sr))

run_demo(AUDIO_URL)